# Pathology region screening — CONCH → MedGemma

Ranks H&E regions by pathological content, so the ones worth putting in a figure next to
TPAF can be picked deliberately instead of by eye. **Screening, not evidence**: the output
selects where to look, it is not a finding in itself.

## Workflow this notebook serves

1. **CONCH scores the whole H&E slide** (this notebook) → ranked regions per structure
2. MedGemma re-reads the shortlist for wording and a second opinion
3. Locally: `path_report.py` → heatmaps + `summary.csv` with full-resolution coordinates
4. **You pick regions, then generate vHE on demand** for just those — TPAF exists for all
   179 FOVs, so virtual staining is run where the figure needs it rather than pre-computed

Step 4 is why `slide` is the primary set: vHE currently covers only **9.7%** of the scored
tissue (7 stitched FOVs, 1.03 mm² of 10.6 mm²), so screening inside the existing vHE would
throw away most of the slide.

## Tile sets in `240817_colab.zip`

| Set | Source | Tiles | µm/px | Role |
|---|---|---|---|---|
| **`slide`** | `WSI_240817HOC240827-4_2_rotate.tif` | **617** | **0.484** | primary — screen the whole slide |
| `vHE` | `stitched_vHE_overlap`, 7 FOV | 28 | 0.621 | optional — how a picked region looks in virtual H&E |
| `realHE` | `_st`, same 7 FOV, registered | 28 | 0.621 | optional — paired counterpart |

All cut at **256 µm**, so scores are comparable across sets.

> ⚠️ **The WSI montage is 0.484 µm/px, not 0.242.** It is a 2× downsample of the per-FOV
> H&E captures (0.242, 40×). Verified by locating three separate H&E FOV TIFs inside the
> montage — all peak at scale 0.48–0.52, none at 1.0. The 0.242 figure and the
> `1/0.39 = 2.564` TPAF:H&E ratio in `notebook_AF_HE_registration.ipynb` both refer to the
> FOV captures, and carrying them over to the montage doubles every tile silently.

> ⚠️ `vHE` / `realHE` come from identical coordinates, but **2 of the 7 FOV are not
> registered** (Line-4_0004, Line-4_0006 — cross-correlation 0.03 vs 0.15–0.52). Their 8
> tiles carry `paired_ok=0`. Fine for scoring one modality alone, invalid as a pair.

## Run order

| Cell | What | Cost |
|---|---|---|
| 1–3 | setup, data, HF login | — |
| **4** | **MedGemma schema check on 6 tiles** | **~1 min** |
| 5 | CONCH scores `slide` (617 tiles × 18 structures) | few min |
| 5b | *optional* — same for `vHE` / `realHE` | <1 min |
| 6 | MedGemma on the CONCH shortlist | tens of min |
| 7 | download |  |

Cell 4 exists because the one thing never tested offline is whether MedGemma's free text
actually parses into the 18 keys. It is deliberately tiny — check it before spending an
hour on cell 6.

**Runtime → T4 is enough.** MedGemma 1.5 4B is ~8 GB in bf16, no quantisation needed.

In [ ]:
#@title 1. Environment
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install --upgrade transformers accelerate huggingface_hub
!pip -q install git+https://github.com/mahmoodlab/CONCH.git
print('ok')

In [ ]:
#@title 2. Data  (240817_colab.zip, ~330 MB — Drive is the sane route at this size)
import os, zipfile

ZIP = '240817_colab.zip'
if not os.path.exists(ZIP):
    # Put the zip anywhere in your Drive; adjust the path below if it is not at the root.
    from google.colab import drive
    drive.mount('/content/drive')
    src = f'/content/drive/MyDrive/{ZIP}'
    if os.path.exists(src):
        os.symlink(src, ZIP)
    else:
        print(f'{src} not found — falling back to browser upload (slow at 330 MB)')
        from google.colab import files
        files.upload()

zipfile.ZipFile(ZIP).extractall('.')

for d, want in (('slide', 617), ('vHE', 28), ('realHE', 28)):
    n = len(os.listdir(f'{d}/tiles'))
    src = dict(l.split('=', 1) for l in open(f'{d}/source.txt').read().splitlines() if '=' in l)
    flag = '' if n == want else f'  <-- expected {want}'
    print(f"{d:8} {n:4} tiles  @ {src['um_per_px']} um/px  tile {src['tile_px']} px "
          f"= {src['tile_um']} um{flag}")

In [ ]:
#@title 3. Hugging Face login  (gated: MahmoodLab/conch, google/medgemma-1.5-4b-it)
from huggingface_hub import login, whoami
login()  # paste a READ token
print(whoami()['name'])

---
## 4. Short run first — MedGemma schema check (6 tiles)

The only untested link in the chain. `run_medgemma` parses lines of the form `key: score`;
if the model prefaces its answer with prose, renames a key, or writes `3/4` instead of `3`,
the parser silently returns all zeros — which looks like a successful run.

**Read the raw text printed below before running cell 6.** What to check:

- all 18 keys present, spelled as in `path_structures.KEYS`
- scores are bare integers 0–4
- no refusal or diagnostic hedging (the prompt says *do not diagnose the patient*)
- `parsed_nonzero` is not 0 for every tile

If the format is off, fix `VLM_INSTRUCTION` in `path_structures.py` and rerun this cell —
not cell 6.

In [ ]:
#@title 4. MedGemma schema check — 6 tiles, ~1 min
import csv, sys
sys.path.insert(0, '.')
from path_structures import KEYS
import path_colab_score as sc

probe = [r['tile_id'] for r in csv.DictReader(open('slide/index.csv'))][::27][:6]
rows = sc.run_medgemma('slide', probe, 'google/medgemma-1.5-4b-it')

for r in rows:
    nz = sum(1 for k in KEYS if float(r[k]) > 0)
    print(f"\n=== {r['tile_id']}  parsed_nonzero={nz}/18")
    print('   raw:', r['raw'][:400])
print('\n--- verdict ---')
tot = sum(sum(1 for k in KEYS if float(r[k]) > 0) for r in rows)
print(f'{tot} non-zero scores across {len(rows)} tiles',
      '-> parser works' if tot else '-> PARSER FAILED, do not run cell 6')

In [ ]:
#@title 5. CONCH on the whole slide — 617 tiles, one forward pass covers all 18 structures
!python path_colab_score.py --tiles slide --backend conch --out conch_slide.csv

import pandas as pd
from path_structures import KEYS, ECM_KEYS
df = pd.read_csv('conch_slide.csv')
print(f'{df.shape[0]} tiles x {len(KEYS)} structures\n')

# Raw similarities, NOT z-scores. path_report.py z-normalises per structure within one
# CSV -- correct for "where on this slide", wrong for comparing sets, since subtracting
# each set's own mean removes exactly the between-set difference. Keeping raw values
# leaves that a post-processing choice with no rescoring needed.
print('per-structure spread (raw cosine similarity):')
s = df[KEYS].agg(['mean', 'std']).T.round(4)
s['ecm'] = [k in ECM_KEYS for k in s.index]
print(s.sort_values('std', ascending=False).to_string())

In [ ]:
#@title 5b. *Optional* — CONCH on vHE / realHE (only if comparing the 7 existing FOV)
# Not needed for the on-demand workflow: you pick regions from the slide ranking, then
# generate vHE for those. Run this only to look at the 7 FOV that already have vHE.
RUN_5B = False  #@param {type:"boolean"}

if RUN_5B:
    for d in ('vHE', 'realHE'):
        !python path_colab_score.py --tiles {d} --backend conch --out conch_{d}.csv
    print('\nreminder: drop the paired_ok=0 tiles before treating these as a matched pair,')
    print('and pool the two CSVs before any normalisation -- per-set z-scores cannot')
    print('answer whether virtual staining preserved a structure.')
else:
    print('skipped')

In [ ]:
#@title 6. MedGemma on the CONCH shortlist  (resumable — rerun after a disconnect)
# Shortlist is taken per structure, so a rare finding still gets looked at instead of
# being buried under whichever structure scores high everywhere. On this slide the
# per-structure top-60 and the overall top-60 share only 18 tiles.
!python path_colab_score.py --tiles slide --backend medgemma \
    --shortlist conch_slide.csv --top 60 --out medgemma_slide.csv

In [ ]:
#@title 7. Download
import os
from google.colab import files
for f in ('conch_slide.csv', 'conch_vHE.csv', 'conch_realHE.csv', 'medgemma_slide.csv'):
    if os.path.exists(f):
        files.download(f)

print('then locally, in UTOM-master:')
print('  python path_report.py --tiles results/path_screen/240817_HE_slide \\')
print('      --scores conch_slide.csv --out results/path_screen/240817_slide_report')
print('  python path_report.py --tiles results/path_screen/240817_vHE_pairs/vHE \\')
print('      --scores conch_vHE.csv --out results/path_screen/240817_vHE_report --only_ecm')
print()
print('--only_ecm keeps the 5 stromal structures where TPAF has a physical reason to')
print('differ from H&E. For vHE vs realHE, pool the two CSVs before normalising --')
print('per-set z-scores cannot answer whether virtual staining preserved a structure,')
print('and drop the paired_ok=0 tiles (Line-4_0004, Line-4_0006) from that comparison.')